# 12 — SCD Tipo 3 — DuckDB

Histórico limitado: colunas `CurrentCity`/`PreviousCity`/`CityChangedOn`.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Carga inicial DimCustomerSCD3
# ============================================================
conn.execute("""
    CREATE OR REPLACE TABLE gold.DimCustomerSCD3 AS
    SELECT
        CustomerID,
        CompanyName,
        ContactName,
        City      AS CurrentCity,
        NULL::VARCHAR AS PreviousCity,
        NULL::DATE    AS CityChangedOn,
        Country   AS CurrentCountry,
        NULL::VARCHAR AS PreviousCountry,
        NULL::DATE    AS CountryChangedOn,
        NOW()         AS LoadTimestamp
    FROM bronze.Customers
""")

n = conn.execute("SELECT COUNT(*) AS n FROM gold.DimCustomerSCD3").fetchdf()['n'][0]
assert n == 91, f"Esperado 91, got {n}"
print(f"✓ DimCustomerSCD3: {n} clientes")


✓ DimCustomerSCD3: 91 clientes


In [3]:
# ============================================================
# SIMULAÇÃO: Modificar bronze para demonstrar SCD3
# (dados sintéticos — Northwind é estático)
# ============================================================
print("--- Estado ANTES ---")
conn.execute("""
    SELECT CustomerID, CurrentCity, PreviousCity, CityChangedOn
    FROM gold.DimCustomerSCD3
    WHERE CustomerID IN ('ALFKI', 'ANATR', 'BOLID')
""").fetchdf()


--- Estado ANTES ---


,CustomerID,CurrentCity,PreviousCity,CityChangedOn
0,ALFKI,Berlin,None,None
1,ANATR,México D.F.,None,None
2,BOLID,Madrid,None,None


In [4]:
# Atualizar bronze (simulação)
conn.execute("UPDATE bronze.Customers SET City = 'Lyon'     WHERE CustomerID = 'ALFKI'")
conn.execute("UPDATE bronze.Customers SET City = 'Madrid'   WHERE CustomerID = 'ANATR'")
conn.execute("UPDATE bronze.Customers SET City = 'Valencia' WHERE CustomerID = 'BOLID'")

# Aplicar lógica SCD3
conn.execute("""
    UPDATE gold.DimCustomerSCD3 scd3
    SET PreviousCity = scd3.CurrentCity,
        CityChangedOn = CURRENT_DATE,
        CurrentCity = b.City
    FROM bronze.Customers b
    WHERE b.CustomerID = scd3.CustomerID
      AND COALESCE(b.City, '') != COALESCE(scd3.CurrentCity, '')
""")

print("--- Estado DEPOIS ---")
result = conn.execute("""
    SELECT CustomerID, CurrentCity AS CidadeAtual,
           PreviousCity AS CidadeAnterior,
           CityChangedOn AS DataMudanca
    FROM gold.DimCustomerSCD3
    WHERE CustomerID IN ('ALFKI', 'ANATR', 'BOLID')
""").fetchdf()
print(result)

# Restaurar bronze
conn.execute("UPDATE bronze.Customers SET City = 'Berlin'       WHERE CustomerID = 'ALFKI'")
conn.execute("UPDATE bronze.Customers SET City = 'México D.F.'  WHERE CustomerID = 'ANATR'")
conn.execute("UPDATE bronze.Customers SET City = 'Madrid'        WHERE CustomerID = 'BOLID'")
print("Bronze restaurado")


--- Estado DEPOIS ---
  CustomerID CidadeAtual CidadeAnterior DataMudanca
0      ALFKI        Lyon         Berlin  2026-03-29
1      ANATR      Madrid    México D.F.  2026-03-29
2      BOLID    Valencia         Madrid  2026-03-29
Bronze restaurado


In [5]:
# ============================================================
# DEMO: Clientes com histórico de mudança
# ============================================================
conn.execute("""
    SELECT CustomerID, CompanyName,
           CurrentCity AS CidadeAtual,
           PreviousCity AS CidadeAnterior,
           CityChangedOn AS MudouEm
    FROM gold.DimCustomerSCD3
    WHERE PreviousCity IS NOT NULL
    ORDER BY CityChangedOn DESC
""").fetchdf()


,CustomerID,CompanyName,CidadeAtual,CidadeAnterior,MudouEm
0,ALFKI,Alfreds Futterkiste,Lyon,Berlin,2026-03-29
1,ANATR,Ana Trujillo Emparedados y helados,Madrid,México D.F.,2026-03-29
2,BOLID,Bólido Comidas preparadas,Valencia,Madrid,2026-03-29
